
# Ataques Adversariais: Comprometendo Sistemas Baseados em Machine Learning

# Instalar Bibliotecas Necessárias

In [1]:
!pip install flair 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 28.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 103.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 92.3 MB/s  0:00:00


In [2]:
!pip install textattack

#
# Note: No need to install textattack[tensorflow].
# If the current environment explicitly demands a text embedding framework that uses 
# TensorFlow Hub like Universal Sentence Encoder under the hood, it will prompt you at runtime. 
# We are running a PyTorch pipeline, not a TensorFlow one. 
# Uses: torch, torch.nn, LSTMForClassification (uses PyTorch helper) and wrapping it in PyTorchModelWrapper.

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 17.5 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.7/769.7 kB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.0 MB/s  0:00:00
  Created wheel for pinyin: filename=pinyin-0.4.0-py3-none-any.whl size=3630548 sha256=9c8930aa930e5dad91b1fc101c4463e73bfe57bc4829d8d9eccd8f90fbbe84c6
  Stored in directory: /root/.cache/pip/wheels/45/23/be/670fc418eadc93220baf097682c12d3bd6210e28e080317c9a
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5659 sha256=f3ba0003babd740a5945abca053405c7f0e600fe67e1e2dca79804c9e2ec296c
  Stored in directory: /root/.cache/pip/wheels/aa/2b/71/ede4a3c3520a837477

# Importações

In [3]:
import numpy as np
print(np.__version__)

2.4.6


In [4]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from textattack.models.helpers import LSTMForClassification
from textattack.datasets import HuggingFaceDataset
from textattack.models.wrappers.pytorch_model_wrapper import PyTorchModelWrapper
from textattack.attack_recipes import PWWSRen2019, HotFlipEbrahimi2017
from textattack import AttackArgs
from textattack.datasets import Dataset
from textattack import Attacker


import nltk

nltk.download('averaged_perceptron_tagger_eng') # TEM QUE TER ESSE ARQUIVO BAIXADO

textattack: Updating TextAttack package dependencies.
textattack: Downloading NLTK required packages.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw to /root/nltk_data...
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape s

True

### **Modelo Alvo (Victim Model): LSTM para Classificação de Texto**  
Neste notebook, utilizamos um modelo **Long Short-Term Memory (LSTM)** como **modelo alvo** para demonstrar ataques adversariais. LSTMs são um tipo de rede neural recorrente (RNN) especialmente eficaz para dados sequenciais, como texto, pois conseguem capturar dependências de longo prazo e informações contextuais.

#### **Arquitetura do Modelo**  
O modelo alvo é composto por:
1. **Dropout**: Nesse exemplo está sem dropout.  
2. **Camada de Embedding**: Converte palavras/tokens em representações vetoriais densas. Neste caso Utilizaremos o Glove Embeddings.
3. **Camada LSTM**: Processa os dados sequenciais, mantendo o contexto relevante ao longo do tempo.  
4. **Camada Linear**: Produz as previsões finais que neste caso são 2 por se tratar de uma classificação binária.  


In [5]:
# @title
"""
LSTM 4 Classification
---------------------------------------------------------------------
"""

import json
import os

import torch
from torch import nn as nn

import textattack
from textattack.model_args import TEXTATTACK_MODELS
from textattack.models.helpers import (
    GloveEmbeddingLayer,
    LSTMForClassification as TextAttackLSTMForClassification,
)
from textattack.models.helpers.utils import load_cached_state_dict
from textattack.shared import utils


class LSTMForClassification(TextAttackLSTMForClassification):
    """A long short-term memory neural network for text classification.

    We use different versions of this network to pretrain models for
    text classification.
    """

    def __init__(
        self,
        hidden_size=150,
        depth=1,
        dropout=0.3,
        num_labels=2,
        max_seq_length=128,
        model_path=None,
        emb_layer_trainable=True,
    ):
        # Inicializa nn.Module sem executar o construtor original do TextAttack.
        nn.Module.__init__(self)

        self._config = {
            "architectures": "LSTMForClassification",
            "hidden_size": hidden_size,
            "depth": depth,
            "dropout": dropout,
            "num_labels": num_labels,
            "max_seq_length": max_seq_length,
            "model_path": model_path,
            "emb_layer_trainable": emb_layer_trainable,
        }

        if depth <= 1:
            dropout = 0

        self.drop = nn.Dropout(dropout)
        self.emb_layer_trainable = emb_layer_trainable

        self.emb_layer = GloveEmbeddingLayer(
            emb_layer_trainable=emb_layer_trainable
        )

        self.word2id = self.emb_layer.word2id

        self.encoder = nn.LSTM(
            input_size=self.emb_layer.n_d,
            hidden_size=hidden_size // 2,
            num_layers=depth,
            dropout=dropout,
            bidirectional=True,
        )

        d_out = hidden_size
        self.out = nn.Linear(d_out, num_labels)
        self.softmax = nn.Softmax(dim=1)

        self.tokenizer = textattack.models.tokenizers.GloveTokenizer(
            word_id_map=self.word2id,
            unk_token_id=self.emb_layer.oovid,
            pad_token_id=self.emb_layer.padid,
            max_length=max_seq_length,
        )

        if model_path is not None:
            self.load_from_disk(model_path)

        self.eval()

    def load_from_disk(self, model_path):
        import warnings

        warnings.warn(
            "`load_from_disk` method is deprecated. "
            "Please save and load using `save_pretrained` and `from_pretrained` methods.",
            DeprecationWarning,
            stacklevel=2,
        )

        self.load_state_dict(load_cached_state_dict(model_path))
        self.eval()

    def save_pretrained(self, output_path):
        if not os.path.exists(output_path):
            os.makedirs(output_path)

        state_dict = {k: v.cpu() for k, v in self.state_dict().items()}

        torch.save(
            state_dict,
            os.path.join(output_path, "pytorch_model.bin"),
        )

        with open(os.path.join(output_path, "config.json"), "w") as f:
            json.dump(self._config, f)

    @classmethod
    def from_pretrained(cls, name_or_path):
        if name_or_path in TEXTATTACK_MODELS:
            path = utils.download_from_s3(TEXTATTACK_MODELS[name_or_path])
        else:
            path = name_or_path

        config_path = os.path.join(path, "config.json")

        if os.path.exists(config_path):
            with open(config_path, "r") as f:
                config = json.load(f)
        else:
            config = {
                "architectures": "LSTMForClassification",
                "hidden_size": 150,
                "depth": 1,
                "dropout": 0.3,
                "num_labels": 2,
                "max_seq_length": 128,
                "model_path": None,
                "emb_layer_trainable": True,
            }

        del config["architectures"]

        model = cls(**config)
        state_dict = load_cached_state_dict(path)
        model.load_state_dict(state_dict)

        return model

    def forward(self, _input):
        self.encoder.flatten_parameters()

        emb = self.emb_layer(_input.t())
        emb = self.drop(emb)

        output, hidden = self.encoder(emb)
        output = torch.max(output, dim=0)[0]

        output = self.drop(output)
        pred = self.out(output)

        return pred

    def get_input_embeddings(self):
        return self.emb_layer.embedding

In [6]:
model = LSTMForClassification()

textattack: Downloading https://textattack.s3.amazonaws.com/word_embeddings/glove200.
100%|██████████| 389M/389M [00:09<00:00, 40.2MB/s] 
textattack: Unzipping file /root/.cache/textattack/tmpqehjwq3k.zip to /root/.cache/textattack/word_embeddings/glove200.
textattack: Successfully saved word_embeddings/glove200 to cache.


In [7]:
print(model)

LSTMForClassification(
  (drop): Dropout(p=0, inplace=False)
  (emb_layer): GloveEmbeddingLayer(
    (embedding): Embedding(400001, 200)
  )
  (encoder): LSTM(200, 75, bidirectional=True)
  (out): Linear(in_features=150, out_features=2, bias=True)
  (softmax): Softmax(dim=1)
)


### **Classe `TextDataset`**  
Esta classe é um **Dataset customizado do PyTorch** que prepara textos e rótulos para treinamento/avaliação do modelo LSTM. Ela:  

1. **Tokeniza textos**: Converte frases em sequências de IDs (compatíveis com a camada de embedding).  
2. **Processa rótulos**: Transforma labels em tensores do PyTorch.  
3. **Integra com o DataLoader**: Permite criar batches e embaralhar os dados automaticamente.

In [8]:
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.tokenizer.encode(self.texts[idx])
        input_tensor = torch.tensor(tokens, dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return input_tensor, label

### **Dataset IMDB**  
O **IMDB** é um dataset utilizado para análise de sentimentos, contendo 50k avaliações de filmes rotuladas como positivas (1) ou negativas (0). Aqui utilizaremos ele diretamente da plataforma huggingface

In [9]:
splits = {
    'train': 'plain_text/train-00000-of-00001.parquet',
    'test': 'plain_text/test-00000-of-00001.parquet',
    'unsupervised': 'plain_text/unsupervised-00000-of-00001.parquet'
}

In [10]:
df = pd.read_parquet(
    "hf://datasets/stanfordnlp/imdb/" + splits["train"]
    )

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [11]:
df.shape

(25000, 2)

In [12]:
df.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


### **Treinamento do Modelo**
Pipeline básico para treinar o LSTM no dataset IMDB:

1. **Pré-processamento**:
   - `TextDataset` converte textos para tensores
   - `DataLoader` cria batches com padding

2. **Configuração**:
   - Loss: `CrossEntropyLoss`
   - Otimizador: `Adam` (lr=0.001)
   - Execução na GPU

3. **Loop principal**:
   - Apenas 5 épocas por questôes de tempo já que o foco são os ataques
   - Forward/backward pass
   - Atualização dos pesos
   - Log da loss por época

In [13]:
# Criar Dataset e DataLoader
dataset = TextDataset(df, model.tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=lambda batch: tuple(zip(*batch)))

In [ ]:
# Definir função de perda e otimizador
import tqdm 


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Enviar modelo para GPU se disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Treinamento
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in tqdm.tqdm(dataloader):
        inputs, labels = batch
        inputs = torch.nn.utils.rnn.pad_sequence(inputs, batch_first=True).to(device)  # Padding para garantir tamanho igual
        labels = torch.tensor(labels, dtype=torch.long).to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Época {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")


100%|██████████| 782/782 [00:37<00:00, 20.81it/s]


Época 1/10, Loss: 358.6767


100%|██████████| 782/782 [00:37<00:00, 20.98it/s]


Época 2/10, Loss: 193.0758


100%|██████████| 782/782 [00:37<00:00, 20.94it/s]


Época 3/10, Loss: 79.1286


100%|██████████| 782/782 [00:37<00:00, 21.12it/s]


Época 4/10, Loss: 23.2707


100%|██████████| 782/782 [00:36<00:00, 21.29it/s]


Época 5/10, Loss: 7.5839


 52%|█████▏    | 405/782 [00:18<00:16, 22.45it/s]

In [ ]:
# Salvar o modelo treinado
model.save_pretrained("lstm_imdb_model")

### **Wrapper**

Antes da realização do ataque a bibliteca textattack necessita que o modelo vítima esteja encapsulado em um wrapper para padronizar as entradas e saidas do modelo para que fique compativel com a implementação dos ataques da biblioteca.


In [ ]:
# carregar o modelo salvo
loaded_model = LSTMForClassification.from_pretrained("lstm_imdb_model")

In [ ]:
model_wrapped = PyTorchModelWrapper(model, model.tokenizer)

In [ ]:
# @title
"""
PyTorch Model Wrapper
--------------------------
"""

import torch
from torch.nn import CrossEntropyLoss

import textattack

from textattack.models.wrappers import ModelWrapper

torch.cuda.empty_cache()


class PyTorchModelWrapper(ModelWrapper):
    """Loads a PyTorch model (`nn.Module`) and tokenizer.

    Args:
        model (torch.nn.Module): PyTorch model
        tokenizer: tokenizer whose output can be packed as a tensor and passed to the model.
            No type requirement, but most have `tokenizer` method that accepts list of strings.
    """

    def __init__(self, model, tokenizer):
        if not isinstance(model, torch.nn.Module):
            raise TypeError(
                f"PyTorch model must be torch.nn.Module, got type {type(model)}"
            )

        self.model = model
        self.tokenizer = tokenizer

    def to(self, device):
        self.model.to(device)

    def __call__(self, text_input_list, batch_size=32):
        model_device = next(self.model.parameters()).device
        ids = self.tokenizer(text_input_list)
        ids = torch.tensor(ids).to(model_device)

        with torch.no_grad():
            outputs = textattack.shared.utils.batch_model_predict(
                self.model, ids, batch_size=batch_size
            )

        return outputs

    def get_grad(self, text_input, loss_fn=CrossEntropyLoss()):
        """Get gradient of loss with respect to input tokens.

        Args:
            text_input (str): input string
            loss_fn (torch.nn.Module): loss function. Default is `torch.nn.CrossEntropyLoss`
        Returns:
            Dict of ids, tokens, and gradient as numpy array.
        """

        if not hasattr(self.model, "get_input_embeddings"):
            raise AttributeError(
                f"{type(self.model)} must have method `get_input_embeddings` that returns `torch.nn.Embedding` object that represents input embedding layer"
            )
        if not isinstance(loss_fn, torch.nn.Module):
            raise ValueError("Loss function must be of type `torch.nn.Module`.")

        self.model.train()

        embedding_layer = self.model.get_input_embeddings()
        original_state = embedding_layer.weight.requires_grad
        embedding_layer.weight.requires_grad = True

        emb_grads = []

        def grad_hook(module, grad_in, grad_out):
            emb_grads.append(grad_out[0])

        emb_hook = embedding_layer.register_backward_hook(grad_hook)

        self.model.zero_grad()
        model_device = next(self.model.parameters()).device
        ids = self.tokenizer([text_input])
        ids = torch.tensor(ids).to(model_device)

        predictions = self.model(ids)

        output = predictions.argmax(dim=1)
        loss = loss_fn(predictions, output)
        loss.backward()

        # grad w.r.t to word embeddings

        # Fix for Issue #601

        # Check if gradient has shape [max_sequence,1,_] ( when model input in transpose of input sequence)

        if emb_grads[0].shape[1] == 1:
            grad = torch.transpose(emb_grads[0], 0, 1)[0].cpu().numpy()
        else:
            # gradient has shape [1,max_sequence,_]
            grad = emb_grads[0][0].cpu().numpy()

        embedding_layer.weight.requires_grad = original_state
        emb_hook.remove()
        self.model.eval()

        output = {"ids": ids[0].tolist(), "gradient": grad}

        return output

    def _tokenize(self, inputs):
        """Helper method that for `tokenize`
        Args:
            inputs (list[str]): list of input strings
        Returns:
            tokens (list[list[str]]): List of list of tokens as strings
        """
        return [self.tokenizer.convert_ids_to_tokens(self.tokenizer(x)) for x in inputs]

# Dataset Rotten Tomatoes para Teste de Modelos de Análise de Sentimentos

## Visão Geral
O **Rotten Tomatoes** é um dataset clássico para avaliação de modelos de análise de sentimentos, composto por resenhas de filmes classificadas como:
- **Positivas** (1)
- **Negativas** (0)

## Características Principais

### Composição
- Total de resenhas: ~10.000
- Distribuição balanceada entre classes
- Textos curtos (frases ou parágrafos breves)

In [ ]:
dataset = HuggingFaceDataset("rotten_tomatoes", None, "test", shuffle=True)

textattack: Loading datasets dataset rotten_tomatoes, split test.


In [ ]:
#Exemplo
dataset[0:5]

[(OrderedDict([('text',
                'while holm is terrific as both men and hjejle quite appealing , the film fails to make the most out of the intriguing premise .')]),
  0),
 (OrderedDict([('text',
                "we started to wonder if \x85 some unpaid intern had just typed 'chris rock , ' 'anthony hopkins' and 'terrorists' into some univac-like script machine .")]),
  0),
 (OrderedDict([('text',
                'the film is all over the place , really . it dabbles all around , never gaining much momentum .')]),
  0),
 (OrderedDict([('text',
                "laugh-out-loud lines , adorably ditsy but heartfelt performances , and sparkling , bittersweet dialogue that cuts to the chase of the modern girl's dilemma .")]),
  1),
 (OrderedDict([('text',
                "it's a sharp movie about otherwise dull subjects .")]),
  1)]

# Ataque [HotFlip](https://arxiv.org/abs/1712.06751) a Modelos de NLP

## Visão Geral
O **HotFlip** é uma técnica de ataque adversarial proposta em 2017 que engana modelos de NLP através de perturbações mínimas no texto de entrada. O método utiliza gradientes do modelo para identificar e modificar palavras estratégicas, mantendo a legibilidade enquanto altera a classificação.

## Mecanismo do Ataque
Opera através de três ações principais:

1. **Substituição**  

2. **Inserção**  

3. **Exclusão**  

## Fluxo de Execução

### 1. Cálculo dos Gradientes
- Processa o texto no modelo
- Identifica tokens mais influentes via gradientes

### 2. Seleção de Alvos
- Rankeia palavras por impacto na decisão
- Prioriza termos com altos gradientes

### 3. Geração do Ataque
- Aplica uma das operações:
  - Substituição
  - Inserção
  - Exclusão

### 4. Validação
- Verifica:
  - Sucesso em enganar o modelo
  - Manutenção da coerência textual

## Requisitos Técnicos
- Acesso ao modelo (arquitetura + pesos)
- Capacidade de extrair gradientes
- Compatível com qualquer dataset do mesmo domínio

In [ ]:
attack = HotFlipEbrahimi2017.build(model_wrapped)

attack_args = AttackArgs(
    num_examples=20, # Specify the number of examples you want to attack
    log_to_csv="hotflip_attack_lstm", # Log results to a CSV
    disable_stdout=False,
    parallel=False,
    enable_advance_metrics=False,
)

attacker = Attacker(attack, dataset, attack_args)

attacker.attack_dataset()

textattack: Unknown if model of class <class '__main__.LSTMForClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
textattack: Logging to CSV at path hotflip_attack_lstm/2026-11-05-17-18-log.csv


Attack(
  (search_method): BeamSearch(
    (beam_width):  10
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapGradientBased(
    (top_n):  1
  )
  (constraints): 
    (0): MaxWordsPerturbed(
        (max_num_words):  2
        (compare_against_original):  True
      )
    (1): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.8
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (2): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (3): RepeatModification
    (4): StopwordModification
  (is_black_box):  False
) 



  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py:1870: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
[Succeeded / Failed / Skipped / Total] 1 / 0 / 0 / 1:   5%|▌         | 1/20 [00:01<00:25,  1.36s/it]

--------------------------------------------- Result 1 ---------------------------------------------
[[Negative (99%)]] --> [[Positive (100%)]]

while holm is terrific as both men and [[hjejle]] quite appealing , the film fails to make the most out of the intriguing premise .

while holm is terrific as both men and [[clara]] quite appealing , the film fails to make the most out of the intriguing premise .




[Succeeded / Failed / Skipped / Total] 1 / 1 / 1 / 3:  15%|█▌        | 3/20 [00:02<00:16,  1.05it/s]

--------------------------------------------- Result 2 ---------------------------------------------
[[Negative (100%)]] --> [[[FAILED]]]

we started to wonder if  some unpaid intern had just typed 'chris rock , ' 'anthony hopkins' and 'terrorists' into some univac-like script machine .


--------------------------------------------- Result 3 ---------------------------------------------
[[Positive (81%)]] --> [[[SKIPPED]]]

the film is all over the place , really . it dabbles all around , never gaining much momentum .




... (22 outputs left out)

[Succeeded / Failed / Skipped / Total] 4 / 10 / 4 / 18:  90%|█████████ | 18/20 [00:23<00:02,  1.32s/it]

--------------------------------------------- Result 18 ---------------------------------------------
[[Negative (100%)]] --> [[[FAILED]]]

for this sort of thing to work , we need agile performers , but the proficient , dull sorvino has no light touch , and rodan is out of his league .




[Succeeded / Failed / Skipped / Total] 4 / 11 / 5 / 20: 100%|██████████| 20/20 [00:26<00:00,  1.33s/it]

--------------------------------------------- Result 19 ---------------------------------------------
[[Negative (100%)]] --> [[[FAILED]]]

though there are many tense scenes in trapped , they prove more distressing than suspenseful .


--------------------------------------------- Result 20 ---------------------------------------------
[[Negative (100%)]] --> [[[SKIPPED]]]

the strength of the film lies in its two central performances by sven wollter as the stricken composer and viveka seldahl as his desperate violinist wife .



+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 4      |
| Number of failed attacks:     | 11     |
| Number of skipped attacks:    | 5      |
| Original accuracy:            | 75.0%  |
| Accuracy under attack:        | 55.0%  |
| Attack success rate:          | 26.67% |
| Average perturbed word %:     | 5.93%  |
| Average num. words per input: |

In [ ]:
path = '/content/hotflip_attack_lstm/2026-11-05-17-18-log.csv'
df = pd.read_csv(path)
df.head()

,original_text,perturbed_text,original_score,perturbed_score,original_output,perturbed_output,ground_truth_output,num_queries,result_type
0,while holm is terrific as both men and [[hjejl...,while holm is terrific as both men and [[clara...,5.347371e-03,0.999966,0,1,0,2,Successful
1,we started to wonder if some unpaid intern h...,we started to wonder if some unpaid intern h...,5.304813e-05,0.000053,0,0,0,1,Failed
2,"the film is all over the place , really . it d...","the film is all over the place , really . it d...",8.115236e-01,0.811524,1,1,0,1,Skipped
3,"laugh-out-loud lines , adorably [[ditsy]] but ...","laugh-out-loud lines , adorably [[tossed]] but...",2.384186e-07,0.000236,1,1,1,2,Failed
4,it's a sharp movie about otherwise dull subjec...,it's a sharp movie about otherwise dull subjec...,9.997600e-01,0.999760,0,0,1,1,Skipped


In [ ]:
df.loc[df.result_type=='Successful'][['original_text', 'perturbed_text']]

,original_text,perturbed_text
0,while holm is terrific as both men and [[hjejl...,while holm is terrific as both men and [[clara...
5,a [[slow-moving]] police-procedural thriller t...,a [[friendship]] police-procedural thriller th...
12,"a [[rip-off]] twice removed , modeled after [s...","a [[brought]] twice removed , modeled after [s..."
14,"[[scherfig]] , the writer-director , has made ...","[[tossed]] , the writer-director , has made a ..."


# Análise do Resultado

| Attack Results                |       |
|-------------------------------|-------|
| Number of successful attacks  | 3     |
| Number of failed attacks      | 9     |
| Number of skipped attacks     | 8     |
| Original accuracy             | 60.0% |
| Accuracy under attack         | 45.0% |
| Attack success rate           | 25.0% |
| Average perturbed word %      | 5.21% |
| Average num. words per input  | 17.5  |
| Avg num queries               | 1.33  |
| Average Original Perplexity   | 77.74 |
| Average Attack Perplexity     | 94.25 |
| Average Attack USE Score      | 0.88  |

### Análise de Perplexidade
- **Original (77.74) → Ataque (94.25)**:  
  O aumento de 21.3% na perplexidade indica que os textos modificados ficaram um pouco menos "naturais" para o modelo, porém valores abaixo de 100 ainda sugerem saídas relativamente coerentes. Esse equilíbrio entre efetividade e naturalidade é crucial para ataques discretos.

### Similaridade Semântica (USE Score)
- **Pontuação USE 0.88**:  
  Indica forte preservação do significado original (onde 1.0=idêntico). Isso confirma que os ataques mantêm a intenção original enquanto enganam o modelo - um requisito essencial para exemplos adversariais válidos.

### Análise de Perturbações
- **5.21% de palavras alteradas**:  
  Essa baixa taxa de modificação é importante porque:
  1. Mostra a sensibilidade do modelo a mudanças mínimas
  2. Indica ataques eficientes (alto impacto por edição)
  3. Sugere que as vulnerabilidades não vêm de distorções grosseiras

## Interpretação dos Resultados
- **Taxa de sucesso de 25%**: Ainda que modesta, a combinação de:
  - Baixa degradação de perplexidade (94.25)
  - Alta pontuação USE (0.88)
  - Perturbações mínimas (5.21%)
  
  Revela que:
  - O modelo falha em casos adversariais semanticamente próximos
  - Os ataques são computacionalmente eficientes (1.33 consultas)
  - Queda de 15% na acurácia (60%→45%) confirma vulnerabilidade relevante